# SawitGuard-GNN — **Lapisan 1 / Tahap 1**: mengotaki sawit dan menandai anomali

Notebook ini utuh dan dapat dijalankan dari atas ke bawah. Seluruh logikanya ada di
`anom.py`; notebook memanggilnya, tidak menyalinnya.

**Keluaran Tahap 1:** untuk setiap citra, daftar kotak sawit beserta kelasnya
(`PalmSan` = normal, `PalmAnom` = anomali/tidak sehat) dan koordinat pusatnya.
Tahap 2 (penilaian kesehatan hilir) **bukan** cakupan notebook ini.

## Data

**Oil Palm Tree Detection for Anomaly Identification** — Dominguez Meza & Rituay, Mendeley,
DOI [10.17632/nh7d23dgnw.1](https://doi.org/10.17632/nh7d23dgnw.1), **CC BY 4.0 (sitasi wajib)**.
424 citra UAV nadir RGB 800×600, DJI Phantom 4 Multispectral, **17–30 m AGL**, 2,5 m/s, **Peru**.
549 kotak: 311 `PalmSan` / 238 `PalmAnom` = **43,4% positif**.

Dataset ini dipakai — bukan ds_B — karena pada ds_B kelas `Unhealthy` hanya 1,3% (≈66 pohon unik,
tajuk ~100 px) dan AP50-nya mentok di 0,43–0,47. Itu batas data, bukan batas model.

## Yang WAJIB dibaca sebelum mengutip angka mana pun dari notebook ini

Bagian 5 menjalankan kontrol yang menentukan bacaan seluruh hasil. Hasil terukurnya:

| lengan (resnet18, split sama) | ROC-AUC |
|---|---|
| `crown` — dipotong ke kotak | 0,951 |
| **`context` — tajuk DIHITAMKAN** | **0,896** |
| `full` — citra utuh | 0,925 |

Pengklasifikasi yang **tidak pernah melihat tajuk** mencapai 0,896. Selisih tajuk-lawan-latar
hanya **0,055**. Karena itu AP50 detektor **tidak boleh dibaca sebagai bukti pengenalan kondisi
tajuk** — sebagian besar keterpisahan kedua kelas tersedia tanpa melihat pohonnya, entah karena
pengelompokan stres yang nyata atau karena perbedaan kondisi akuisisi. Data ini tidak dapat
membedakan keduanya.

Selain itu, dataset ini menganotasi **≈1,27 sawit per citra** padahal bingkainya memuat 3–6.
Model di sini karena itu belajar *"kotaki sawit yang akan dipilih penganotasi"*, bukan *"kotaki
setiap sawit"*. Untuk kemampuan kotaki-setiap-sawit, ds_B-lah datasetnya — di sana setiap pohon
dianotasi dan AP50 kelas `Healthy` sudah **0,978 / 0,989** (lihat `LABEL_QUALITY_AUDIT.md` dan
`y12.py`).

## Klaim maksimum yang boleh ditulis

> Detektor memisahkan kedua kelas dataset ini pada AP50 = **X**, tetapi pengklasifikasi yang
> tidak melihat tajuk sama sekali mencapai ROC-AUC 0,896 pada split yang sama; jadi sebagian
> besar keterpisahan itu kontekstual. `PalmAnom` berarti "tertekan atau sakit" menurut penulis
> dataset — **bukan BSR, bukan Ganoderma**, tanpa verifikasi lapangan. Tidak ada metadata situs,
> jadi tidak ada klaim lintas-kebun.

In [ ]:
import os, sys, json, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
sys.path.insert(0, os.getcwd())
import anom

import torch, ultralytics
print("ultralytics", ultralytics.__version__)
print("torch      ", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("sumber data:", anom.DS)
assert os.path.isdir(anom.DS), "Dataset tidak ditemukan - lihat docstring anom.py"

## 0. Setelan

Ubah di sini saja.

`FOLDS` mengendalikan biaya. `["fold0"]` = satu holdout terstratifikasi 339/86, cepat (~5 menit
pada RTX 5060) tetapi **tanpa pita derau**. Daftar penuh `None` = 5 lipatan, memberi mean ± std
yang dibutuhkan naskah (~76 menit). Untuk angka yang dikutip di paper, pakai 5 lipatan.

In [ ]:
MODEL   = "yolo12n.pt"     # yolo12s.pt bila ada waktu/VRAM
EPOCHS  = 30               # 30 sudah mendekati jenuh pada 424 citra
IMGSZ   = 640
K       = 5                # jumlah lipatan yang DIBANGUN
FOLDS   = ["fold0"]        # yang DILATIH; None = semua K lipatan
SEEDS   = (42,)
CONF    = 0.25
CACHE   = "ram"
WORKERS = 0                # WAJIB 0 di notebook Windows (spawn dataloader)
TAG     = "stage1"

n_run = (K if FOLDS is None else len(FOLDS)) * len(SEEDS)
print("akan melatih %d lari x %d epoch  (~%.0f menit @ ~9 dtk/epoch)"
      % (n_run, EPOCHS, n_run * EPOCHS * 9 / 60))
print("hasil disimpan per-lari; notebook boleh dihentikan lalu dilanjutkan.")

## 1. Audit dataset — sebelum melatih apa pun

Angka mutu dulu, hasil belakangan. Ini menentukan metrik apa yang sah dilaporkan.

In [ ]:
info = anom.audit()

### Audit kebocoran

Sudah dijalankan saat dataset diadopsi, dan hasilnya **bersih** — berbeda dari ds_B:

| pemeriksaan | hasil |
|---|---|
| berkas identik byte-per-byte | **0** |
| pasangan mirip (Hamming ≤ 20/256, pHash) di dalam maupun antar-split | **0** |
| 20 "stem" nama berkas yang muncul di >1 split | korelasi piksel 0,007–0,085 ⇒ citra berbeda, penomoran Roboflow bertabrakan — **bukan kebocoran** |
| luas kotak → kelas (ROC-AUC) | 0,422 ⇒ geometri **bukan** jalan pintas |
| rasio aspek → kelas | 0,514 ⇒ tidak informatif |

Karena tidak ada duplikasi piksel, seluruh 424 citra boleh disatukan lalu dibagi ulang dengan
k-fold terstratifikasi. Split bawaan (338/22/64) diabaikan: 22 citra validasi terlalu sedikit
untuk pita derau yang berarti. **Ini keputusan yang berlawanan dengan ds_B, dengan alasan yang
sama** — ikuti struktur kebocoran datanya, bukan kebiasaan.

In [ ]:
folds = anom.build(k=K)
run_folds = FOLDS if FOLDS else folds
print("\nakan dilatih:", run_folds)

## 2. Latih detektor 2-kelas

Satu model mengerjakan keduanya sekaligus: mengotaki sawit **dan** memberinya kelas
`PalmSan`/`PalmAnom`. Itulah bentuk keluaran Tahap 1.

In [ ]:
t0 = time.time()
res = anom.train_cv(model=MODEL, folds=run_folds, epochs=EPOCHS, imgsz=IMGSZ,
                    seeds=SEEDS, cache=CACHE, workers=WORKERS, tag=TAG)
print("\nselesai dalam %.1f menit" % ((time.time() - t0) / 60))

## 3. Hasil

`PalmAnom AP50` adalah angka utamanya. Bila hanya satu lipatan dijalankan, `std` akan `nan` —
itu benar, bukan galat: satu angka tidak punya simpangan baku.

In [ ]:
anom.report(TAG)

## 4. Kerapatan anotasi — mengapa presisi di sini adalah batas bawah

In [ ]:
import glob
W = sorted(glob.glob(os.path.join(anom.RUNS, "%s_%s_s%d" % (TAG, run_folds[0], SEEDS[0]),
                                  "weights", "best.pt")))[0]
print("bobot:", os.path.relpath(W, anom.BASE))
dens = anom.annotation_density(W, fold=run_folds[0], conf=CONF, imgsz=IMGSZ)

## 5. KONTROL — berapa banyak sinyalnya ada di tajuk, berapa di latar

Bagian terpenting notebook ini. Tiga pengklasifikasi pada split yang sama:

- `crown` — dipotong ke kotak GT, jadi **hanya tajuk**
- `context` — kotak GT **dihitamkan**, jadi **hanya latar; pohonnya dihapus**
- `full` — citra utuh

Bila `context` mendekati `crown`, maka AP50 detektor sebagian besar bukan tentang kondisi
tajuk. Laporkan hasil dataset ini **relatif terhadap `context`**, bukan terhadap 0,5.

In [ ]:
ctl = anom.controls(fold=run_folds[0], epochs=15)

## 6. Prediksi kualitatif

Perhatikan sawit yang jelas menguning tetapi **tidak dianotasi** — itu bukan kesalahan model,
melainkan kerapatan anotasi dataset (bagian 4).

In [ ]:
import matplotlib.pyplot as plt
fig = anom.qualitative(W, fold=run_folds[0], n=8, conf=CONF, imgsz=IMGSZ)
plt.show()

## 7. Ringkasan dan klaim maksimum

Merakit kalimat yang boleh ditulis, dari angka yang benar-benar ada. Bila sebuah angka tidak
muncul di sini, ia tidak boleh muncul di naskah.

In [ ]:
d = json.load(open(os.path.join(anom.RESDIR, TAG + ".json")))
keys = sorted(d["runs"])
g = lambda fn: np.array([fn(d["runs"][k]) for k in keys], float)
sd = lambda a: float(a.std(ddof=1)) if len(a) > 1 else float("nan")
anomap = g(lambda r: r["ap50"].get("PalmAnom", np.nan))
sanap = g(lambda r: r["ap50"].get("PalmSan", np.nan))
m50, m95 = g(lambda r: r["map50"]), g(lambda r: r["map"])
st = d["setting"]

lines = [
 "TAHAP 1 - mengotaki sawit dan menandai anomali.",
 "",
 "Data: Dominguez Meza & Rituay, DOI 10.17632/nh7d23dgnw.1, CC BY 4.0. 424 citra UAV",
 "nadir RGB (Peru), 800x600, DJI Phantom 4 Multispectral, 17-30 m AGL. 549 kotak,",
 "43,4%% positif. Evaluasi: %d lipatan terstratifikasi x %d seed, %s %d epoch imgsz %d."
 % (len(st["folds"]), len(st["seeds"]), st["model"], st["epochs"], st["imgsz"]),
 "",
 "HASIL",
 "  PalmAnom AP50 = %.3f +/- %.3f" % (np.nanmean(anomap), sd(anomap)),
 "  PalmSan  AP50 = %.3f +/- %.3f" % (np.nanmean(sanap), sd(sanap)),
 "  mAP50         = %.3f +/- %.3f" % (np.nanmean(m50), sd(m50)),
 "  mAP50-95      = %.3f +/- %.3f" % (np.nanmean(m95), sd(m95)),
 "",
 "KONTROL YANG WAJIB IKUT DIKUTIP",
 "  hanya tajuk (crown)   ROC-AUC = %.3f" % ctl["crown"]["roc_auc"],
 "  TANPA tajuk (context) ROC-AUC = %.3f   <-- garis dasar sesungguhnya"
 % ctl["context"]["roc_auc"],
 "  citra utuh (full)     ROC-AUC = %.3f" % ctl["full"]["roc_auc"],
 "  selisih crown - context = %.3f"
 % (ctl["crown"]["roc_auc"] - ctl["context"]["roc_auc"]),
 "",
 "BATAS YANG MELEKAT",
 "1. Pengklasifikasi yang tidak pernah melihat tajuk mencapai %.3f. AP50 di atas"
 % ctl["context"]["roc_auc"],
 "   karena itu BUKAN bukti pengenalan kondisi tajuk; sebagian besar keterpisahan",
 "   kedua kelas bersifat kontekstual. Data ini tidak dapat membedakan apakah itu",
 "   pengelompokan stres yang nyata atau perbedaan kondisi akuisisi.",
 "2. 'PalmAnom' = 'tertekan atau sakit' menurut penulis dataset. BUKAN BSR, BUKAN",
 "   Ganoderma, tanpa verifikasi lapangan.",
 "3. Anotasi hanya %.2f kotak per citra padahal bingkai memuat 3-6 sawit. Model ini"
 % dens["gt_per_image"],
 "   belajar mengotaki sawit yang DIPILIH penganotasi, bukan setiap sawit. Presisi",
 "   terhadap GT ini adalah batas bawah. Untuk kotaki-setiap-sawit gunakan ds_B",
 "   (AP50 Healthy 0,978/0,989, lihat LABEL_QUALITY_AUDIT.md).",
 "4. Tidak ada metadata situs, jadi block-CV per-situs mustahil. Yang dapat dijamin",
 "   hanya ketiadaan duplikat piksel. Tidak ada klaim lintas-kebun.",
]
if len(keys) < 2:
    lines += ["5. HANYA %d lipatan dijalankan: angka di atas TIDAK punya pita derau." % len(keys),
              "   Jalankan FOLDS=None sebelum mengutip di naskah."]
print("=" * 78); print("\n".join(lines)); print("=" * 78)

json.dump(dict(setting=st, folds_run=keys,
               palmanom_ap50=[float(np.nanmean(anomap)), sd(anomap)],
               palmsan_ap50=[float(np.nanmean(sanap)), sd(sanap)],
               map50=[float(np.nanmean(m50)), sd(m50)],
               map50_95=[float(np.nanmean(m95)), sd(m95)],
               controls=ctl, annotation_density=dens,
               source_doi="10.17632/nh7d23dgnw.1", licence="CC BY 4.0"),
          open(os.path.join(anom.BASE, "stage1_summary.json"), "w"), indent=2, default=float)
print("tersimpan: stage1_summary.json")

## 8. Ekspor model — `stage1_model.pkl`

Satu berkas mandiri berisi bobot **dan** metadata. Isinya `dict` biasa, bukan objek kelas
khusus, sehingga dapat di-`pickle.load` di mesin mana pun **tanpa** `anom.py`. Bobotnya
dibenamkan sebagai bytes `.pt`, jadi berkasnya tidak merujuk path lokal apa pun.

Batas pembacaannya ikut dibenamkan di `payload["limits"]` — termasuk angka kontrol. Ini
disengaja: angka model tidak boleh beredar terpisah dari syarat pembacaannya, dan siapa pun
yang memuat berkas ini mendapatkan caveat-nya sekaligus.

In [ ]:
PKL = anom.export_pickle(W, tag=TAG)

In [ ]:
# Uji muat-ulang. Inilah cara Tahap 2 memakainya.
model, meta = anom.load_pickle(PKL)
demo = [l.strip() for l in open(os.path.join(anom.ROOT, "%s_val.txt" % run_folds[0]))
        if l.strip()][:3]
for src, r in zip(demo, model.predict(demo, imgsz=meta["imgsz"], conf=meta["conf"],
                                      verbose=False)):
    b = r.boxes
    xy = b.xywh.cpu().numpy()
    print(os.path.basename(src)[:44])
    for (cx, cy, bw, bh), c, k in zip(xy, b.conf.cpu().numpy(), b.cls.cpu().numpy().astype(int)):
        print("   %-9s conf=%.2f  pusat=(%.0f, %.0f)  kotak=%.0fx%.0f"
              % (meta["names"][k], c, cx, cy, bw, bh))
print("\nBATAS yang ikut terbawa di dalam berkas:")
for i, t in enumerate(meta["limits"], 1):
    print("  %d. %s" % (i, t))

### Memuat tanpa repositori ini

```python
import pickle, tempfile, os
from ultralytics import YOLO

d = pickle.load(open("stage1_model.pkl", "rb"))
p = os.path.join(tempfile.mkdtemp(), "w.pt")
open(p, "wb").write(d["weights_pt"])

model = YOLO(p)
res = model.predict("citra.jpg", imgsz=d["imgsz"], conf=d["conf"])
# d["names"] -> {0: 'PalmSan', 1: 'PalmAnom'}
# d["limits"] -> syarat pembacaan angka model ini
```

Butuh `ultralytics` terpasang; versi yang dipakai saat ekspor tercatat di `d["framework"]`.